# Kaggle One-File Runner

只上传这个 `.ipynb` 到 Kaggle 即可运行。

支持两种获取项目代码方式（优先级从上到下）：
1. 从 `PROJECT_ZIP_INPUT` 解压（如果你把项目 zip 作为 Kaggle Dataset Input 添加）
2. 从 `REPO_URL` git clone（无需上传代码文件）


In [ ]:
# ====== 你只需要改这里 ======
PROJECT_ZIP_INPUT = ""  # 例如: /kaggle/input/project-code/project_for_kaggle.zip
REPO_URL = ""           # 例如: https://github.com/yourname/yourrepo.git
REPO_BRANCH = "main"

PROJECT_DIR = "/kaggle/working/project"

# 数据路径（注意：是 /kaggle/input/<dataset-slug>/...，不是网页 URL）
CASIA_ROOT = "/kaggle/input/data_pic/data/raw/CASIA2"
NIST_ROOT = ""  # 没有 NIST 就留空

# 训练参数
MODEL = "dual_stream_lite"
BATCH_SIZE = 2
EPOCHS = 30


In [ ]:
import os
import shutil
import zipfile
import subprocess
from pathlib import Path

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

if PROJECT_ZIP_INPUT and Path(PROJECT_ZIP_INPUT).exists():
    print(f"[Code Source] unzip: {PROJECT_ZIP_INPUT}")
    with zipfile.ZipFile(PROJECT_ZIP_INPUT, "r") as zf:
        zf.extractall(project_dir)

    # 如果 zip 内还有一层根目录，自动进入该目录
    children = [p for p in project_dir.iterdir() if p.is_dir()]
    if len(children) == 1 and (children[0] / "requirements.txt").exists():
        project_dir = children[0]
elif REPO_URL:
    print(f"[Code Source] git clone: {REPO_URL} (branch={REPO_BRANCH})")
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(project_dir)
    ])
else:
    raise RuntimeError("请至少设置 PROJECT_ZIP_INPUT 或 REPO_URL 其中一个。")

print("Project dir:", project_dir)
if not (project_dir / "requirements.txt").exists():
    raise FileNotFoundError(f"requirements.txt 不存在: {project_dir}")

%cd {project_dir}


In [ ]:
# 安装依赖（保留 Kaggle 预装 torch，避免破坏 GPU 环境）
import re
from pathlib import Path
import subprocess
import sys

req_text = Path("requirements.txt").read_text(encoding="utf-8")
lines = []
for raw in req_text.splitlines():
    line = raw.strip()
    if not line or line.startswith("#"):
        continue
    pkg = re.split(r"[<>=!~]", line, maxsplit=1)[0].strip().lower()
    if pkg in {"torch", "torchvision", "torchaudio"}:
        continue
    lines.append(line)

Path("/kaggle/working/requirements-kaggle.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "/kaggle/working/requirements-kaggle.txt"])


In [ ]:
# GPU 检查
!python check_gpu.py


In [ ]:
# 训练前路径检查（自动兜底常见 Kaggle 路径）
from pathlib import Path

casia_candidates = [
    CASIA_ROOT,
    "/kaggle/input/data_pic/data/raw/CASIA2",
    "/kaggle/input/data-pic/data/raw/CASIA2",
    "/kaggle/input/data_pic/CASIA2",
    "/kaggle/input/data-pic/CASIA2",
]
for p in casia_candidates:
    if p and Path(p).exists():
        CASIA_ROOT = p
        break

print("CASIA exists:", Path(CASIA_ROOT).exists(), CASIA_ROOT)
if NIST_ROOT:
    print("NIST  exists:", Path(NIST_ROOT).exists(), NIST_ROOT)
else:
    print("NIST  exists:", False, "(skip)")


In [ ]:
# 启动训练
import os
import sys
import subprocess
from pathlib import Path

required = [Path("src"), Path("src/data"), Path("src/models"), Path("src/train.py")]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "仓库缺少训练代码文件: " + ", ".join(missing) +
        "。请把这些目录推送到 GitHub 后再 Run All。"
    )

os.environ["PYTHONPATH"] = str(Path.cwd())
cmd = [
    sys.executable, "-m", "src.train",
    "--model", MODEL,
    "--casia-root", CASIA_ROOT,
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--device", "cuda",
]
if NIST_ROOT:
    cmd.extend(["--nist-root", NIST_ROOT])
print("Run:", " ".join(cmd))
subprocess.check_call(cmd)
